### Read

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

claims_schema = StructType([
    StructField("claim_id", StringType(), False),
    StructField("patient_id", StringType(), False),  # PHI
    StructField("provider_id", StringType(), False),
    StructField("diagnosis_code", StringType(), True),
    StructField("procedure_code", StringType(), True),
    StructField("billed_amount", DoubleType(), True),
    StructField("claim_date", StringType(), True),
    StructField("denial_flag", IntegerType(), True),
])

df_raw = spark.read.csv(
    "/Volumes/workspace/default/myvol/raw/claims/claims_1000.csv",
    header=True,
    schema=claims_schema
)
print(f"Raw rows loaded: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

### Add metadata columns

In [0]:
df_bronze = df_raw \
    .withColumn("_ingestion_timestamp", F.current_timestamp()) \
    .withColumn("_source_file", F.lit("claims_1000.csv")) \
    .withColumn("_source_layer", F.lit("bronze")) \
    .withColumn("_is_phi", F.lit(True))  # PHI flag — patient_id present

print("PHI fields: patient_id, claim_id")
print(f"Bronze rows: {df_bronze.count()}")
df_bronze.show(5, truncate=False)

### Write to delta lake

In [0]:
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/myvol/bronze/claims/")

print("✅ Bronze claims Delta table written")

### Validate

In [0]:
df_verify = spark.read.format("delta") \
    .load("/Volumes/workspace/default/myvol/bronze/claims/")

print(f"Verification row count: {df_verify.count()}")
assert df_verify.count() == df_raw.count(), "❌ Row count mismatch after write!"
print(f"✅ Validation passed — {df_verify.count()} rows")
df_verify.printSchema()